# CrewAI Market Research Crew

**Project:** Multi-Agent Market Research with CrewAI  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **multi-agent crew** using [CrewAI](https://docs.crewai.com/) to perform comprehensive market research for a product or service. The crew consists of four specialized agents:

| Agent | Role | Responsibility |
|-------|------|----------------|
| **Market Researcher** | Data Gatherer | Identifies market size, trends, and customer segments |
| **Competitor Analyst** | Competitive Intel | Maps competitors, their strengths, weaknesses, and positioning |
| **Pricing Strategist** | Pricing Expert | Recommends pricing models based on market and competitor data |
| **Critic Reviewer** | Quality Assurance | Reviews the combined analysis for gaps, biases, and actionable insights |

### How CrewAI Works

CrewAI orchestrates multiple AI agents that collaborate on complex tasks:

1. **Agents** — autonomous units with a defined role, goal, and backstory that shapes their behavior
2. **Tasks** — specific assignments given to agents, with descriptions and expected outputs
3. **Crew** — the coordination layer that manages agent execution order, information flow, and final output
4. **Process** — the execution strategy (sequential or hierarchical)

In a **sequential process**, tasks execute in order and each agent can build on the previous agent's output. This creates a pipeline where research feeds into competitor analysis, which feeds into pricing strategy, which is then critically reviewed.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime
from typing import Optional

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

We use **Ollama** running locally so no API keys are needed. CrewAI's `LLM` wrapper connects to the local Ollama server.

In [ ]:
# Configure the local LLM via Ollama
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

Each agent has three key attributes that shape its behavior:

- **`role`** — The job title that anchors the agent's identity
- **`goal`** — What the agent is trying to achieve (drives its decision-making)
- **`backstory`** — Context that gives the agent domain expertise and personality

The backstory is critical — it acts as a system prompt that makes the agent behave like a domain expert rather than a generic assistant.

### 3.1 Market Research Agent

**Purpose:** Gathers foundational market data — market size, growth trajectory, customer segments, and emerging trends. This agent's output sets the stage for all downstream analysis.

In [ ]:
market_researcher = Agent(
    role="Senior Market Research Analyst",
    goal=(
        "Conduct thorough market research to identify total addressable market (TAM), "
        "serviceable addressable market (SAM), growth trends, customer segments, "
        "and key market drivers for the given product or industry."
    ),
    backstory=(
        "You are a veteran market research analyst with 15 years of experience at "
        "top-tier consulting firms like McKinsey and BCG. You specialize in sizing "
        "markets, identifying growth vectors, and segmenting customers. You always "
        "structure your analysis with clear data points and cite specific trends. "
        "You think in frameworks like TAM/SAM/SOM, Porter's Five Forces, and "
        "PESTEL analysis. Your outputs are data-driven and actionable."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {market_researcher.role}")

### 3.2 Competitor Analysis Agent

**Purpose:** Maps the competitive landscape — identifies key players, analyzes their strengths and weaknesses, evaluates market positioning, and spots competitive gaps. Builds on the market researcher's findings.

In [ ]:
competitor_analyst = Agent(
    role="Competitive Intelligence Specialist",
    goal=(
        "Analyze the competitive landscape by identifying key competitors, "
        "their market share, product offerings, strengths, weaknesses, "
        "differentiation strategies, and potential vulnerabilities to exploit."
    ),
    backstory=(
        "You are a competitive intelligence specialist who has spent a decade "
        "tracking market dynamics across tech, SaaS, and consumer industries. "
        "You excel at building competitor matrices, SWOT analyses, and "
        "identifying strategic positioning gaps. You analyze competitors through "
        "multiple lenses: product features, pricing, go-to-market strategy, "
        "customer satisfaction, and technology stack. You always present findings "
        "in structured tables and highlight actionable competitive advantages."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {competitor_analyst.role}")

### 3.3 Pricing Strategist Agent

**Purpose:** Develops pricing recommendations based on market positioning, competitor pricing, cost structures, and customer willingness to pay. Uses the market research and competitor data as inputs.

In [ ]:
pricing_strategist = Agent(
    role="Pricing Strategy Director",
    goal=(
        "Develop a comprehensive pricing strategy with specific price points, "
        "tier structures, and pricing models that maximize revenue while "
        "remaining competitive in the market."
    ),
    backstory=(
        "You are a pricing strategy expert who has designed pricing models for "
        "50+ product launches across SaaS, marketplace, and consumer sectors. "
        "You understand value-based pricing, cost-plus models, freemium strategies, "
        "and dynamic pricing. You always consider price elasticity, competitive "
        "anchoring, psychological pricing, and willingness-to-pay segmentation. "
        "Your recommendations include specific numbers, tier breakdowns, and "
        "revenue projections with clear justification for each price point."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {pricing_strategist.role}")

### 3.4 Critic Reviewer Agent

**Purpose:** Acts as **quality assurance** — reviews the combined output of all previous agents, identifies gaps, biases, weak assumptions, and ensures the final report is coherent, balanced, and actionable. This is the "red team" of the crew.

In [ ]:
critic_reviewer = Agent(
    role="Strategic Analysis Critic",
    goal=(
        "Critically review the combined market research, competitive analysis, "
        "and pricing strategy for logical gaps, unsupported claims, biases, "
        "missing perspectives, and internal contradictions. Produce a final "
        "consolidated report with confidence ratings and risk flags."
    ),
    backstory=(
        "You are a senior strategy partner known for your rigorous analytical "
        "standards. You have reviewed hundreds of market analyses and pricing "
        "proposals. Your job is to stress-test every claim, challenge assumptions, "
        "and ensure the final deliverable is bulletproof. You flag unsupported "
        "assertions, identify risks the team may have overlooked, check for "
        "internal consistency, and rate the confidence level of each section. "
        "You are constructive but uncompromising on quality."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {critic_reviewer.role}")

### Agent Summary

In [ ]:
agents = [market_researcher, competitor_analyst, pricing_strategist, critic_reviewer]

print("=" * 70)
print("CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[Agent {i}] {agent.role}")
    print(f"  Goal: {agent.goal[:80]}...")
    print(f"  Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)

## 4. Define Tasks

Each task is assigned to a specific agent and describes:
- **`description`** — What the agent should do (the prompt)
- **`expected_output`** — The format and content of the deliverable
- **`agent`** — Which agent executes this task

In sequential mode, each task receives the output of the previous task as context, creating a chain of analysis.

### 4.1 Define the Research Target

We parameterize the product/industry so the crew can be reused for any market research project.

In [ ]:
# =====================================================
# CONFIGURE YOUR RESEARCH TARGET HERE
# =====================================================
PRODUCT = "AI-powered code review tool for enterprise development teams"
INDUSTRY = "Developer Tools / DevOps SaaS"
TARGET_MARKET = "Mid-to-large enterprise software companies (500+ developers)"
# =====================================================

print(f"Research Target: {PRODUCT}")
print(f"Industry: {INDUSTRY}")
print(f"Target Market: {TARGET_MARKET}")

### 4.2 Task 1 — Market Research

In [ ]:
task_market_research = Task(
    description=(
        f"Conduct comprehensive market research for: {PRODUCT}\n"
        f"Industry: {INDUSTRY}\n"
        f"Target Market: {TARGET_MARKET}\n\n"
        "Your analysis MUST cover:\n"
        "1. **Market Size**: Estimate TAM, SAM, and SOM with reasoning\n"
        "2. **Growth Trends**: Key growth drivers and projected CAGR\n"
        "3. **Customer Segments**: Primary, secondary, and emerging segments\n"
        "4. **Market Drivers**: Technology, regulatory, and demand drivers\n"
        "5. **Barriers to Entry**: Technical, financial, and market barriers\n"
        "6. **Key Risks**: Market risks and mitigation strategies\n"
    ),
    expected_output=(
        "A structured market research report with sections for market sizing, "
        "growth analysis, customer segmentation, drivers, barriers, and risks. "
        "Include specific numbers and estimates where possible."
    ),
    agent=market_researcher,
)

print(f"Task created: Market Research -> {task_market_research.agent.role}")

### 4.3 Task 2 — Competitor Analysis

In [ ]:
task_competitor_analysis = Task(
    description=(
        f"Analyze the competitive landscape for: {PRODUCT}\n"
        f"Industry: {INDUSTRY}\n\n"
        "Using the market research provided by the previous analyst, perform:\n"
        "1. **Competitor Identification**: List 5-8 key competitors (direct and indirect)\n"
        "2. **Competitor Matrix**: Compare features, pricing, target audience, market share\n"
        "3. **SWOT Analysis**: For the top 3 competitors\n"
        "4. **Positioning Map**: Where competitors sit on key dimensions\n"
        "5. **Competitive Gaps**: Unserved or underserved needs in the market\n"
        "6. **Threat Assessment**: Which competitors pose the biggest threat and why\n"
    ),
    expected_output=(
        "A competitive intelligence report with competitor profiles, a comparison "
        "matrix, SWOT analyses, positioning insights, identified gaps, and a "
        "threat ranking. Use structured tables where appropriate."
    ),
    agent=competitor_analyst,
)

print(f"Task created: Competitor Analysis -> {task_competitor_analysis.agent.role}")

### 4.4 Task 3 — Pricing Strategy

In [ ]:
task_pricing_strategy = Task(
    description=(
        f"Develop a pricing strategy for: {PRODUCT}\n"
        f"Target Market: {TARGET_MARKET}\n\n"
        "Using the market research AND competitive analysis from previous agents:\n"
        "1. **Pricing Model**: Recommend the best model (subscription, usage-based, "
        "   per-seat, hybrid) with justification\n"
        "2. **Tier Structure**: Define 3-4 pricing tiers with features per tier\n"
        "3. **Price Points**: Specific monthly/annual prices for each tier\n"
        "4. **Competitive Positioning**: How pricing compares to competitors\n"
        "5. **Revenue Projections**: Estimated ARR at different adoption scenarios\n"
        "6. **Pricing Psychology**: Anchoring, decoy effects, and value framing\n"
        "7. **Discounting Strategy**: Volume discounts, annual vs monthly, enterprise deals\n"
    ),
    expected_output=(
        "A pricing strategy document with a recommended pricing model, detailed "
        "tier breakdown with specific prices, competitive price positioning, "
        "revenue projections, and a discounting framework."
    ),
    agent=pricing_strategist,
)

print(f"Task created: Pricing Strategy -> {task_pricing_strategy.agent.role}")

### 4.5 Task 4 — Critical Review

In [ ]:
task_critic_review = Task(
    description=(
        "You have received the combined outputs of three specialist agents:\n"
        "- Market Research (TAM/SAM, trends, segments)\n"
        "- Competitive Analysis (competitor profiles, SWOT, gaps)\n"
        "- Pricing Strategy (model, tiers, price points, projections)\n\n"
        "Perform a critical review:\n"
        "1. **Consistency Check**: Are the market size, competitor, and pricing "
        "   claims internally consistent?\n"
        "2. **Assumption Audit**: Identify unsupported claims or weak assumptions\n"
        "3. **Gap Analysis**: What perspectives or data points are missing?\n"
        "4. **Risk Assessment**: Rate overall risk (Low/Medium/High) with justification\n"
        "5. **Confidence Ratings**: Rate each section's reliability (1-5 stars)\n"
        "6. **Recommendations**: Provide 3-5 specific improvements or next steps\n"
        "7. **Executive Summary**: Synthesize everything into a concise summary "
        "   that a C-suite executive could read in 2 minutes\n"
    ),
    expected_output=(
        "A critical review report with consistency findings, flagged assumptions, "
        "gap analysis, risk ratings, confidence scores per section, actionable "
        "recommendations, and a 2-minute executive summary."
    ),
    agent=critic_reviewer,
)

print(f"Task created: Critical Review -> {task_critic_review.agent.role}")

### Task Pipeline Overview

In [ ]:
tasks = [task_market_research, task_competitor_analysis, task_pricing_strategy, task_critic_review]

print("TASK PIPELINE (Sequential)")
print("=" * 60)
for i, task in enumerate(tasks, 1):
    suffix = "FINAL OUTPUT" if i == len(tasks) else f"feeds into Task {i+1}"
    print(f"  Task {i}: {task.agent.role}")
    print(f"         {suffix}")
    if i < len(tasks):
        print(f"         |")
print("=" * 60)

## 5. Assemble and Run the Crew

### How Crew Coordination Works

The `Crew` object is the **orchestrator** that manages:

1. **Agent Sequencing** — In `Process.sequential`, tasks execute in list order. Each agent completes its task before the next begins.
2. **Context Passing** — The output of each task is automatically passed as context to the next task. This means the competitor analyst sees the market researcher's findings, the pricing strategist sees both, and the critic sees everything.
3. **Memory** — When `memory=True`, the crew maintains a shared memory that agents can reference across tasks.
4. **Verbose Logging** — Shows the agent's internal reasoning (chain-of-thought), tool usage, and final answers.

```
+--------------------+     +--------------------+     +--------------------+     +--------------------+
|  Market Research   |---->| Competitor Intel    |---->| Pricing Strategy   |---->|  Critic Review     |
|     Agent          |     |     Agent           |     |     Agent          |     |     Agent          |
+--------------------+     +--------------------+     +--------------------+     +--------------------+
       TAM/SAM                    SWOT                   Price Tiers             Executive Summary
       Segments                Competitors                Revenue                Confidence Ratings
       Trends                     Gaps                   Discounts                 Risk Flags
```

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,  # Tasks run in order, output flows forward
    verbose=True,                # Show agent reasoning
    memory=False,                # Disable for deterministic runs; enable for multi-run sessions
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")
print(f"Memory: {'Enabled' if crew.memory else 'Disabled'}")

In [ ]:
# Execute the crew
print("=" * 70)
print(f"LAUNCHING CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Target: {PRODUCT}")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Consolidated Report

In [ ]:
# Display the final output (from the critic reviewer)
print("=" * 70)
print("FINAL REPORT — Critical Review & Executive Summary")
print("=" * 70)
print(result.raw)

### 6.2 Individual Task Outputs

Each task's output is preserved separately, so we can inspect what each agent produced.

In [ ]:
task_names = ["Market Research", "Competitor Analysis", "Pricing Strategy", "Critical Review"]

for name, task in zip(task_names, tasks):
    print("\n" + "=" * 70)
    print(f"TASK OUTPUT: {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        # Show first 1500 chars to keep display manageable
        output_text = task.output.raw
        if len(output_text) > 1500:
            print(output_text[:1500])
            print(f"\n... [{len(output_text) - 1500} more characters]")
        else:
            print(output_text)
    else:
        print("[No output captured]")

### 6.3 Token Usage & Performance

In [ ]:
# Display token usage if available
if hasattr(result, 'token_usage') and result.token_usage:
    usage = result.token_usage
    print("TOKEN USAGE SUMMARY")
    print("=" * 40)
    print(f"  Total tokens:      {usage.get('total_tokens', 'N/A')}")
    print(f"  Prompt tokens:     {usage.get('prompt_tokens', 'N/A')}")
    print(f"  Completion tokens: {usage.get('completion_tokens', 'N/A')}")
else:
    print("Token usage not tracked (common with local Ollama models)")

# Output lengths per task
print("\nOUTPUT SIZES")
print("=" * 40)
for name, task in zip(task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    print(f"  {name:25s}: {length:,} chars")

## 7. Save Full Report to Disk

In [ ]:
# Build a complete report combining all task outputs
report_sections = []
report_sections.append(f"# Market Research Report: {PRODUCT}")
report_sections.append(f"**Generated:** {datetime.now().isoformat()}")
report_sections.append(f"**Industry:** {INDUSTRY}")
report_sections.append(f"**Target Market:** {TARGET_MARKET}")
report_sections.append("---")

for name, task in zip(task_names, tasks):
    report_sections.append(f"\n## {name}\n")
    report_sections.append(f"**Agent:** {task.agent.role}\n")
    report_sections.append(task.output.raw if task.output else "[No output]")
    report_sections.append("\n---")

full_report = "\n".join(report_sections)

report_path = "market_research_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(full_report)

print(f"Full report saved to: {report_path}")
print(f"Report size: {len(full_report):,} characters")

## 8. Experiment: Different Product Target

The crew is fully reusable — change the product and re-run to research any market.

In [ ]:
# =====================================================
# SECOND RESEARCH TARGET
# =====================================================
PRODUCT_2 = "AI-powered personal finance assistant mobile app"
INDUSTRY_2 = "FinTech / Personal Finance"
TARGET_MARKET_2 = "Millennials and Gen Z (ages 22-40) in North America"

print(f"Second Research Target: {PRODUCT_2}")
print(f"Industry: {INDUSTRY_2}")
print(f"Target Market: {TARGET_MARKET_2}")

In [ ]:
# Create new tasks with the second product target
task2_market = Task(
    description=(
        f"Conduct comprehensive market research for: {PRODUCT_2}\n"
        f"Industry: {INDUSTRY_2}\n"
        f"Target Market: {TARGET_MARKET_2}\n\n"
        "Cover: TAM/SAM/SOM, growth trends, customer segments, "
        "market drivers, barriers to entry, and key risks."
    ),
    expected_output="Structured market research report with data-driven estimates.",
    agent=market_researcher,
)

task2_competitor = Task(
    description=(
        f"Analyze competitors for: {PRODUCT_2}\n"
        f"Industry: {INDUSTRY_2}\n\n"
        "Identify 5-8 competitors, build a comparison matrix, SWOT for top 3, "
        "and identify competitive gaps."
    ),
    expected_output="Competitive intelligence report with profiles, SWOT, and gaps.",
    agent=competitor_analyst,
)

task2_pricing = Task(
    description=(
        f"Develop pricing strategy for: {PRODUCT_2}\n"
        f"Target Market: {TARGET_MARKET_2}\n\n"
        "Recommend pricing model, define 3-4 tiers with prices, compare to "
        "competitors, and project revenue scenarios."
    ),
    expected_output="Pricing document with tiers, specific prices, and revenue projection.",
    agent=pricing_strategist,
)

task2_critic = Task(
    description=(
        "Review the combined market research, competitor analysis, and pricing "
        "strategy. Check consistency, flag weak assumptions, rate confidence, "
        "and produce a 2-minute executive summary."
    ),
    expected_output="Critical review with confidence ratings, risk flags, and exec summary.",
    agent=critic_reviewer,
)

print("Second set of tasks created for FinTech target")

In [ ]:
# Assemble and run crew for the second target
crew2 = Crew(
    agents=agents,
    tasks=[task2_market, task2_competitor, task2_pricing, task2_critic],
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"\nLaunching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Target: {PRODUCT_2}")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Display second research final output
print("=" * 70)
print(f"FINAL REPORT — {PRODUCT_2}")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Research Runs

In [ ]:
# Side-by-side comparison
print("RESEARCH COMPARISON")
print("=" * 70)
print(f"{'Metric':<30} {'DevTools Research':<20} {'FinTech Research':<20}")
print("-" * 70)

tasks1 = [task_market_research, task_competitor_analysis, task_pricing_strategy, task_critic_review]
tasks2_list = [task2_market, task2_competitor, task2_pricing, task2_critic]

for name, t1, t2 in zip(task_names, tasks1, tasks2_list):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{name + ' (chars)':<30} {len1:<20,} {len2:<20,}")

total1 = sum(len(t.output.raw) for t in tasks1 if t.output)
total2 = sum(len(t.output.raw) for t in tasks2_list if t.output)
print("-" * 70)
print(f"{'Total output':<30} {total1:<20,} {total2:<20,}")

## 10. Deep Dive: Crew Coordination Patterns

### Sequential vs Hierarchical Processes

CrewAI supports two process types:

| Feature | Sequential | Hierarchical |
|---------|-----------|-------------|
| Execution | Linear, task-by-task | Manager agent delegates dynamically |
| Context flow | Output of task N → input of task N+1 | Manager passes context selectively |
| Best for | Predictable pipelines | Complex, adaptive workflows |
| Overhead | Lower | Higher (extra manager agent reasoning) |

### Agent Design Principles

1. **Specificity**: Give agents narrow, well-defined roles. A "do everything" agent produces worse results than specialized agents.
2. **Backstory matters**: The backstory is effectively a system prompt — it shapes the agent's reasoning style, expertise areas, and output format.
3. **Delegation control**: Setting `allow_delegation=False` prevents agents from passing work to others, ensuring each agent completes its own task.
4. **Expected output**: Clear expected output descriptions act as format constraints that improve output quality.

### Why a Critic Agent?

The critic agent serves as a **quality gate**:
- Catches internal contradictions (e.g., market size claims that don't match pricing assumptions)
- Identifies missing perspectives the other agents may have overlooked
- Provides confidence ratings so stakeholders know which claims are strong vs speculative
- Produces an executive summary that synthesizes all findings

## 11. Extending the Crew

Demonstrate how to add a new agent to an existing crew without rebuilding everything.

In [ ]:
# Add a Go-to-Market strategist as a 5th agent
gtm_strategist = Agent(
    role="Go-to-Market Strategist",
    goal=(
        "Design a go-to-market launch plan including channel strategy, "
        "messaging framework, launch timeline, and key success metrics."
    ),
    backstory=(
        "You are a GTM strategist who has launched 30+ B2B SaaS products. "
        "You think in terms of ICP (ideal customer profile), channel mix, "
        "messaging hierarchy, and launch milestones. You create actionable "
        "90-day launch plans with clear owners and KPIs."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

task_gtm = Task(
    description=(
        f"Using all previous research on {PRODUCT}, create a go-to-market plan:\n"
        "1. Ideal Customer Profile (ICP) definition\n"
        "2. Channel strategy (top 5 channels ranked by expected ROI)\n"
        "3. Messaging framework (positioning statement, key messages per segment)\n"
        "4. 90-day launch timeline with milestones\n"
        "5. Success metrics and KPIs\n"
    ),
    expected_output=(
        "A go-to-market plan with ICP, channel ranking, messaging framework, "
        "timeline, and KPIs."
    ),
    agent=gtm_strategist,
)

print(f"New agent created: {gtm_strategist.role}")
print(f"New task created for GTM planning")

In [ ]:
# Assemble extended crew (5 agents, 5 tasks)
extended_agents = agents + [gtm_strategist]
extended_tasks = tasks + [task_gtm]

crew_extended = Crew(
    agents=extended_agents,
    tasks=extended_tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Extended crew: {len(crew_extended.agents)} agents, {len(crew_extended.tasks)} tasks")
print("\nPipeline:")
for i, t in enumerate(extended_tasks, 1):
    print(f"  {i}. {t.agent.role}")

In [ ]:
# Run the extended crew
print(f"Launching extended crew — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

result_extended = crew_extended.kickoff()

print(f"\nExtended crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Display GTM plan (the new agent's output)
print("=" * 70)
print("GO-TO-MARKET PLAN")
print("=" * 70)
print(result_extended.raw)

## 12. Key Takeaways

### What We Built
- A **4-agent crew** (Market Research → Competitor Analysis → Pricing → Critic Review) that produces a comprehensive market analysis
- Extended it to **5 agents** by adding a Go-to-Market strategist
- Ran the crew on **two different products** to demonstrate reusability

### CrewAI Concepts Demonstrated
1. **Agent Roles & Backstories** — How role/goal/backstory shapes agent behavior and output quality
2. **Sequential Processing** — Tasks execute in order with automatic context passing
3. **Task Design** — Structured descriptions with numbered requirements produce better outputs
4. **Critic Pattern** — A review agent that validates and synthesizes the work of other agents
5. **Crew Extensibility** — Adding agents and tasks without rebuilding the pipeline
6. **Local LLM** — Running everything locally via Ollama (no API keys needed)

### Production Considerations
- Enable `memory=True` for crews that run repeatedly (accumulates knowledge across runs)
- Add tools (web search, file reading) to give agents access to real data
- Use `Process.hierarchical` when agents need to dynamically decide what to research
- Set `max_rpm` on the LLM config to prevent rate limiting with hosted APIs
- Consider adding `output_json` or `output_pydantic` to tasks for structured, parseable outputs